In [ ]:
!pip install medspacy --quiet
!pip install negspacy --quiet
print("Negation-detection libraries installed.")

In [ ]:
# ===== NOTEBOOK CELL 1: SETUP + REAL DATA PATH =====
__author__ = "S. L. Toriola, MD, MSc."
import os
from pathlib import Path
import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

KAGGLE_DATA = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files and "competitions" in root and "rsna-knee-abnormality-detection" in root:
        KAGGLE_DATA = Path(root)
        break
if KAGGLE_DATA is None:
    raise FileNotFoundError("Could not find real competition train.csv")
print("Found real competition data at:", KAGGLE_DATA)

In [ ]:
# ===== NOTEBOOK CELL 2: TRANSLATION (checks for a pre-translated dataset first) =====
import time
import os
import pandas as pd
from pathlib import Path

output_path = Path("/kaggle/working/train_translated.csv")

# Look for an already-translated dataset you've attached as Input (update this name/path once you've created it)
PRETRANSLATED_DATASET_PATH = None
for root, dirs, files in os.walk("/kaggle/input/datasets"):
    if "train_translated.csv" in files:
        PRETRANSLATED_DATASET_PATH = Path(root) / "train_translated.csv"
        break

if PRETRANSLATED_DATASET_PATH is not None:
    print(f"Found a real pre-translated dataset at {PRETRANSLATED_DATASET_PATH} — skipping translation entirely.")
    train_full = pd.read_csv(PRETRANSLATED_DATASET_PATH)
elif output_path.exists():
    print("train_translated.csv already exists this session — skipping re-translation.")
    train_full = pd.read_csv(output_path)
else:
    print("No pre-translated data found anywhere — running real translation now (this will take a while).")
    !pip install langdetect --quiet
    from langdetect import detect, LangDetectException
    from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

    model_name = "facebook/m2m100_418M"
    translator_tokenizer = M2M100Tokenizer.from_pretrained(model_name)
    translator_model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)
    print("Translation model loaded on:", device)

    def translate_to_english(text, src_lang):
        if src_lang == "en" or not text.strip():
            return text
        try:
            translator_tokenizer.src_lang = src_lang
            encoded = translator_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
            generated = translator_model.generate(
                **encoded, forced_bos_token_id=translator_tokenizer.get_lang_id("en"), max_length=512
            )
            return translator_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
        except Exception as e:
            print(f"Translation failed for one report, keeping original: {e}")
            return text

    train_full = pd.read_csv(KAGGLE_DATA / "train.csv")
    reports = train_full["Report"].astype(str)

    detected_langs = []
    translated_reports = []
    t0 = time.time()
    for i, text in enumerate(reports):
        try:
            lang = detect(text) if text.strip() else "en"
        except LangDetectException:
            lang = "en"
        detected_langs.append(lang)
        translated_reports.append(translate_to_english(text, lang))
        if (i + 1) % 200 == 0:
            print(f"  Translated {i+1}/{len(reports)} reports, {round(time.time()-t0)}s elapsed")

    train_full["detected_language"] = detected_langs
    train_full["Report_English"] = translated_reports
    train_full.to_csv(output_path, index=False)
    print("Saved train_translated.csv")

In [ ]:
# ===== NOTEBOOK CELL 4a: LOAD RADLEX AND BUILD REAL ABNORMAL-FINDING VOCABULARY (fixed) =====
!pip install owlready2 --quiet
import os
from pathlib import Path
from owlready2 import get_ontology

RADLEX_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower() == "radlex.owl":
            RADLEX_PATH = Path(root) / f
            break
    if RADLEX_PATH:
        break

if RADLEX_PATH is None:
    raise FileNotFoundError("RadLex.owl not found under /kaggle/input — check the radlex-owl dataset is attached.")

print(f"Loading RadLex from {RADLEX_PATH} ...")
radlex_onto = get_ontology(f"file://{RADLEX_PATH}").load()

# Pull labels, keeping only English ones where RadLex tags a language,
# and falling back to an ASCII-only check when no language tag is present
# (filters out the German terms that flooded the first attempt).
def is_english(lbl):
    lang = getattr(lbl, "lang", None)
    if lang is not None:
        return lang == "en"
    try:
        lbl.encode("ascii")
        return True
    except UnicodeEncodeError:
        return False

radlex_labels = []
for cls in radlex_onto.classes():
    for lbl in cls.label:
        if is_english(lbl):
            radlex_labels.append(str(lbl))
    if hasattr(cls, "prefLabel"):
        for lbl in cls.prefLabel:
            if is_english(lbl):
                radlex_labels.append(str(lbl))

radlex_labels = list(set(radlex_labels))
print(f"Loaded {len(radlex_labels)} real English-tagged RadLex terms.")

# Fixed keyword filter — ANY RadLex term containing one of these real abnormal-finding
# words is kept, regardless of whether it also names an anatomical location. These get
# combined with your existing anatomy anchors in Cell 4 (e.g. "medial.*?<this term>"),
# not required to already be combined inside a single RadLex label.
ABNORMAL_KEYWORDS = [
    "tear", "torn", "rupture", "sprain", "degeneration", "degenerative",
    "osteoarthritis", "narrowing", "osteophyte", "cartilage loss", "cartilage defect",
    "chondromalacia", "marrow lesion", "cyst", "attrition", "maceration",
    "extrusion", "subluxation", "hypertrophy", "effusion", "synovitis",
    "thickening", "distension", "distention", "contusion", "bruise",
    "fracture", "stress reaction", "insufficiency fracture", "bursitis",
    "chondral", "meniscal injury", "ligament injury", "ligamentous injury",
]

RADLEX_FINDING_TERMS = set()
for label in radlex_labels:
    label_lower = label.lower()
    if any(kw in label_lower for kw in ABNORMAL_KEYWORDS):
        RADLEX_FINDING_TERMS.add(label_lower)

print(f"RadLex abnormal-finding vocabulary (anatomy-agnostic): {len(RADLEX_FINDING_TERMS)} real terms")
print("Sample:", list(RADLEX_FINDING_TERMS)[:20])

In [ ]:
# ===== NOTEBOOK CELL 4: LABEL EXTRACTION — medspaCy + negspacy + RadLex vocabulary =====
import re
import pandas as pd
import spacy
import medspacy
from spacy.util import filter_spans
from negspacy.negation import Negex

# ---- Pipeline 1: medspaCy ConText ----
nlp = medspacy.load(enable=["medspacy_pyrush", "medspacy_context"])
if "sentencizer" not in nlp.pipe_names and "parser" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")
context = nlp.get_pipe("medspacy_context")
from medspacy.context import ConTextRule

custom_context_rules = [
    ConTextRule(literal="likely represents", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="in favor of", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="most consistent with", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="most likely represents", category="PSEUDO", direction="BIDIRECTIONAL"),
]
context.add(custom_context_rules)
test_doc = nlp.make_doc("Bone marrow edema in medial femoral condyle, likely represents bone contusion.")
start = test_doc.text.find("contusion")
span = test_doc.char_span(start, start + len("contusion"), label="Contusion", alignment_mode="expand")
test_doc.ents = [span]
test_doc = nlp.get_pipe("sentencizer")(test_doc)
test_doc = context(test_doc)
ent = test_doc.ents[0]
print("is_negated:", ent._.is_negated, "| is_uncertain:", ent._.is_uncertain, "| modifiers:", list(ent._.modifiers))

nlp.add_pipe("negex", config={"ent_types": None})

MR_KNEE_TEMPLATE_1_RADREPORT = """ ... (paste your existing full text here) ... """
MR_KNEE_TEMPLATE_2_COMPARTMENT = """ ... (paste your existing full text here) ... """

POSITIVE_PATTERNS = {
    "ACL": [
        r"\bACL\b.*?\b(tear|rupture|sprain|injur\w*)\b",
        r"\bACL\b.*?insufficien\w*",
        r"anterior subluxation of the tibia",
    ],
    "MCL": [
        r"\bMCL\b.*?\b(tear|sprain|injur\w*)\b",
        r"\bMCL\b.*?insufficien\w*",
    ],
    "Medial Meniscus": [
        r"medial meniscus.*?\b(tear|degenerat\w*)\b",
        r"medial meniscus.*?(radial tear|parrot-beak tear|horizontal tear|complex tear|"
        r"bucket-handle tear|root tear|macerat\w*|extrusion|subluxation|hypertroph\w*)",
    ],
    "Lateral Meniscus": [
        r"lateral meniscus.*?\b(tear|degenerat\w*)\b",
        r"lateral meniscus.*?(radial tear|parrot-beak tear|horizontal tear|complex tear|"
        r"bucket-handle tear|root tear|macerat\w*|extrusion|subluxation|hypertroph\w*)",
    ],
    "Medial OA": [
        r"medial.*?(osteoarthrit\w*|compartment narrowing)",
        r"medial.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "Lateral OA": [
        r"lateral.*?(osteoarthrit\w*|compartment narrowing)",
        r"lateral.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "PF OA": [
        r"patellofemoral.*?(osteoarthrit\w*)",
        r"patellofemoral.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "Effusion": [
        r"\beffusion\b",
        r"synovial (thickening|distension|distention)",
        r"capsular distention",
    ],
    "Synovitis": [
        r"\bsynovitis\w*\b",
        r"hoffa.?synovitis",
        r"synovial (thickening|distension|distention)",
    ],
    "Baker's": [
        r"baker'?s? cyst",
        r"popliteal cyst",
        r"ganglion cyst",
    ],
    "Contusion": [
        r"\bcontusion\b|\bbone bruis\w*\b",
    ],
    "Fracture": [
        r"\bfractur\w*\b",
        r"stress reaction",
        r"insufficiency fracture",
        r"osteochondral fracture",
    ],
}

# ---- Fold in real RadLex terms (from Cell 4a) paired with anatomy anchors ----
RADLEX_TARGET_KEYWORDS = {
    "ACL": ["cruciate", "acl"],
    "MCL": ["collateral", "mcl"],
    "Medial Meniscus": ["meniscus", "meniscal"],
    "Lateral Meniscus": ["meniscus", "meniscal"],
    "Medial OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "Lateral OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "PF OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "Effusion": ["effusion"],
    "Synovitis": ["synov"],
    "Baker's": ["popliteal", "baker"],
    "Contusion": ["contusion", "bruise"],
    "Fracture": ["fracture"],
}
RADLEX_ANCHORS = {
    "ACL": r"\bACL\b",
    "MCL": r"\bMCL\b",
    "Medial Meniscus": r"medial meniscus",
    "Lateral Meniscus": r"lateral meniscus",
    "Medial OA": r"medial",
    "Lateral OA": r"lateral",
    "PF OA": r"patellofemoral",
    "Effusion": None,
    "Synovitis": None,
    "Baker's": None,
    "Contusion": None,
    "Fracture": None,
}

for target, keywords in RADLEX_TARGET_KEYWORDS.items():
    matching_terms = [t for t in RADLEX_FINDING_TERMS if any(kw in t for kw in keywords)]
    if not matching_terms:
        continue
    escaped = "|".join(re.escape(t) for t in matching_terms)
    anchor = RADLEX_ANCHORS[target]
    if anchor:
        POSITIVE_PATTERNS.setdefault(target, []).append(rf"{anchor}.*?({escaped})")
    else:
        POSITIVE_PATTERNS.setdefault(target, []).append(rf"({escaped})")

_COMPILED = {t: [re.compile(p, re.IGNORECASE) for p in pats] for t, pats in POSITIVE_PATTERNS.items()}

NORMAL_PATTERNS = {
    "ACL": [r"cruciate ligaments?.*?\bintact\b", r"no cruciate.*?ligament tears?", r"\bACL\b.*?\bintact\b"],
    "MCL": [r"\bMCL\b.*?\bintact\b", r"medial collateral ligament.*?intact", r"no cruciate or collateral ligament tears?"],
    "Medial Meniscus": [r"medial meniscus.*?\bintact\b", r"normal menisci", r"medial compartment:?\s*meniscus normal"],
    "Lateral Meniscus": [r"lateral meniscus.*?\bintact\b", r"normal menisci", r"lateral compartment:?\s*meniscus normal"],
    "Medial OA": [r"medial compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
                  r"medial compartment.*?cartilage preserved", r"medial compartment.*?subchondral bone normal"],
    "Lateral OA": [r"lateral compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
                   r"lateral compartment.*?cartilage preserved", r"lateral compartment.*?subchondral bone normal"],
    "PF OA": [r"patellofemoral compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
              r"patellofemoral compartment.*?cartilage preserved", r"patellofemoral compartment.*?subchondral bone normal"],
    "Effusion": [r"no joint effusion", r"\bno effusion\b"],
    "Baker's": [r"no baker'?s? cyst", r"no popliteal cyst"],
    "Contusion": [r"no fracture,?\s*contusion,?\s*(or|and)\s*suspicious lesion"],
    "Fracture": [r"no fracture,? stress reaction,? or osseous lesion", r"no fracture,?\s*contusion,?\s*(or|and)\s*suspicious lesion"],
}
_NORMAL_COMPILED = {t: [re.compile(p, re.IGNORECASE) for p in pats] for t, pats in NORMAL_PATTERNS.items()}
GLOBAL_NORMAL_PATTERNS = [re.compile(r"normal mri of the knee", re.IGNORECASE)]

def extract_labels_from_report(report_text):
    if not isinstance(report_text, str) or not report_text.strip():
        return {t: None for t in POSITIVE_PATTERNS}

    doc = nlp.make_doc(report_text)
    spans = []
    for target, patterns in _COMPILED.items():
        for pattern in patterns:
            for match in pattern.finditer(report_text):
                span = doc.char_span(match.start(), match.end(), label=target, alignment_mode="expand")
                if span is not None:
                    spans.append(span)
    doc.ents = filter_spans(spans)

    doc = nlp.get_pipe("sentencizer")(doc)
    doc = context(doc)
    doc = nlp.get_pipe("negex")(doc)

    is_globally_normal = any(p.search(report_text) for p in GLOBAL_NORMAL_PATTERNS)

    result = {t: None for t in POSITIVE_PATTERNS}
    for ent in doc.ents:
        medspacy_negated = ent._.is_negated or ent._.is_uncertain or ent._.is_hypothetical
        negspacy_negated = ent._.negex
        if not (medspacy_negated or negspacy_negated):
            result[ent.label_] = 1

    for target in NORMAL_PATTERNS:
        if target in result:
            for pattern in _NORMAL_COMPILED[target]:
                if pattern.search(report_text):
                    result[target] = None
                    break

    if is_globally_normal:
        for target in result:
            if result[target] != 1:
                result[target] = None

    return result

train_full_loaded = train_full
labels_df = pd.read_csv(KAGGLE_DATA / "train.csv")

target_columns = list(POSITIVE_PATTERNS.keys())
extracted = train_full_loaded["Report_English"].apply(extract_labels_from_report).apply(pd.Series)

for col in target_columns:
    missing_mask = labels_df[col].isna()
    labels_df.loc[missing_mask, col] = extracted.loc[missing_mask, col]

labels_df[target_columns] = labels_df[target_columns].fillna(0).astype(float)
labels_df = labels_df[["StudyInstanceUID"] + target_columns]
labels_df.to_csv("/kaggle/working/train_labels_filled.csv", index=False)
print("Saved train_labels_filled.csv — regex + RadLex + medspaCy + negspacy.")
print(labels_df[target_columns].sum())

In [ ]:
# ===== NOTEBOOK CELL 4b: FLAG SUSPICIOUS LABELS (matches Cell 4's RadLex + medspaCy + negspacy logic) =====
import re
import pandas as pd
import spacy
import medspacy
from spacy.util import filter_spans
from negspacy.negation import Negex

nlp = medspacy.load(enable=["medspacy_pyrush", "medspacy_context"])
if "sentencizer" not in nlp.pipe_names and "parser" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")
context = nlp.get_pipe("medspacy_context")
from medspacy.context import ConTextRule

custom_context_rules = [
    ConTextRule(literal="likely represents", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="in favor of", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="most consistent with", category="PSEUDO", direction="BIDIRECTIONAL"),
    ConTextRule(literal="most likely represents", category="PSEUDO", direction="BIDIRECTIONAL"),
]
context.add(custom_context_rules)
nlp.add_pipe("negex", config={"ent_types": None})

POSITIVE_PATTERNS = {
    "ACL": [
        r"\bACL\b.*?\b(tear|rupture|sprain|injur\w*)\b",
        r"\bACL\b.*?insufficien\w*",
        r"anterior subluxation of the tibia",
    ],
    "MCL": [
        r"\bMCL\b.*?\b(tear|sprain|injur\w*)\b",
        r"\bMCL\b.*?insufficien\w*",
    ],
    "Medial Meniscus": [
        r"medial meniscus.*?\b(tear|degenerat\w*)\b",
        r"medial meniscus.*?(radial tear|parrot-beak tear|horizontal tear|complex tear|"
        r"bucket-handle tear|root tear|macerat\w*|extrusion|subluxation|hypertroph\w*)",
    ],
    "Lateral Meniscus": [
        r"lateral meniscus.*?\b(tear|degenerat\w*)\b",
        r"lateral meniscus.*?(radial tear|parrot-beak tear|horizontal tear|complex tear|"
        r"bucket-handle tear|root tear|macerat\w*|extrusion|subluxation|hypertroph\w*)",
    ],
    "Medial OA": [
        r"medial.*?(osteoarthrit\w*|compartment narrowing)",
        r"medial.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "Lateral OA": [
        r"lateral.*?(osteoarthrit\w*|compartment narrowing)",
        r"lateral.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "PF OA": [
        r"patellofemoral.*?(osteoarthrit\w*)",
        r"patellofemoral.*?(cartilage loss|cartilage thinning|focal cartilage defect|"
        r"full.thickness loss|partial.thickness loss|chondromalacia|osteophyte\w*|"
        r"marrow lesion|marrow edema|subarticular cyst|subchondral cyst|bone attrition|"
        r"joint space narrowing)",
    ],
    "Effusion": [
        r"\beffusion\b",
        r"synovial (thickening|distension|distention)",
        r"capsular distention",
    ],
    "Synovitis": [
        r"\bsynovitis\w*\b",
        r"hoffa.?synovitis",
        r"synovial (thickening|distension|distention)",
    ],
    "Baker's": [
        r"baker'?s? cyst",
        r"popliteal cyst",
        r"ganglion cyst",
    ],
    "Contusion": [
        r"\bcontusion\b|\bbone bruis\w*\b",
    ],
    "Fracture": [
        r"\bfractur\w*\b",
        r"stress reaction",
        r"insufficiency fracture",
        r"osteochondral fracture",
    ],
}

RADLEX_TARGET_KEYWORDS = {
    "ACL": ["cruciate", "acl"],
    "MCL": ["collateral", "mcl"],
    "Medial Meniscus": ["meniscus", "meniscal"],
    "Lateral Meniscus": ["meniscus", "meniscal"],
    "Medial OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "Lateral OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "PF OA": ["cartilage", "chondromalacia", "osteophyte", "osteoarthrit", "chondral"],
    "Effusion": ["effusion"],
    "Synovitis": ["synov"],
    "Baker's": ["popliteal", "baker"],
    "Contusion": ["contusion", "bruise"],
    "Fracture": ["fracture"],
}
RADLEX_ANCHORS = {
    "ACL": r"\bACL\b",
    "MCL": r"\bMCL\b",
    "Medial Meniscus": r"medial meniscus",
    "Lateral Meniscus": r"lateral meniscus",
    "Medial OA": r"medial",
    "Lateral OA": r"lateral",
    "PF OA": r"patellofemoral",
    "Effusion": None,
    "Synovitis": None,
    "Baker's": None,
    "Contusion": None,
    "Fracture": None,
}

for target, keywords in RADLEX_TARGET_KEYWORDS.items():
    matching_terms = [t for t in RADLEX_FINDING_TERMS if any(kw in t for kw in keywords)]
    if not matching_terms:
        continue
    escaped = "|".join(re.escape(t) for t in matching_terms)
    anchor = RADLEX_ANCHORS[target]
    if anchor:
        POSITIVE_PATTERNS.setdefault(target, []).append(rf"{anchor}.*?({escaped})")
    else:
        POSITIVE_PATTERNS.setdefault(target, []).append(rf"({escaped})")

_COMPILED = {t: [re.compile(p, re.IGNORECASE) for p in pats] for t, pats in POSITIVE_PATTERNS.items()}

NORMAL_PATTERNS = {
    "ACL": [r"cruciate ligaments?.*?\bintact\b", r"no cruciate.*?ligament tears?", r"\bACL\b.*?\bintact\b"],
    "MCL": [r"\bMCL\b.*?\bintact\b", r"medial collateral ligament.*?intact", r"no cruciate or collateral ligament tears?"],
    "Medial Meniscus": [r"medial meniscus.*?\bintact\b", r"normal menisci", r"medial compartment:?\s*meniscus normal"],
    "Lateral Meniscus": [r"lateral meniscus.*?\bintact\b", r"normal menisci", r"lateral compartment:?\s*meniscus normal"],
    "Medial OA": [r"medial compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
                  r"medial compartment.*?cartilage preserved", r"medial compartment.*?subchondral bone normal"],
    "Lateral OA": [r"lateral compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
                   r"lateral compartment.*?cartilage preserved", r"lateral compartment.*?subchondral bone normal"],
    "PF OA": [r"patellofemoral compartment.*?no hyaline cartilage disease", r"no osseous or cartilaginous abnormality",
              r"patellofemoral compartment.*?cartilage preserved", r"patellofemoral compartment.*?subchondral bone normal"],
    "Effusion": [r"no joint effusion", r"\bno effusion\b"],
    "Baker's": [r"no baker'?s? cyst", r"no popliteal cyst"],
    "Contusion": [r"no fracture,?\s*contusion,?\s*(or|and)\s*suspicious lesion"],
    "Fracture": [r"no fracture,? stress reaction,? or osseous lesion", r"no fracture,?\s*contusion,?\s*(or|and)\s*suspicious lesion"],
}
_NORMAL_COMPILED = {t: [re.compile(p, re.IGNORECASE) for p in pats] for t, pats in NORMAL_PATTERNS.items()}
GLOBAL_NORMAL_PATTERNS = [re.compile(r"normal mri of the knee", re.IGNORECASE)]

raw_labels = pd.read_csv(KAGGLE_DATA / "train.csv")
labels_filled = pd.read_csv("/kaggle/working/train_labels_filled.csv")
# train_full already in memory from Cell 2 — do not re-read from disk

target_columns = list(POSITIVE_PATTERNS.keys())
merged = labels_filled.merge(
    train_full[["StudyInstanceUID", "Report_English", "detected_language"]],
    on="StudyInstanceUID", how="left"
)

flags = []
for idx, row in merged.iterrows():
    text = row["Report_English"]
    if not isinstance(text, str) or not text.strip():
        continue

    word_count = len(text.split())
    if word_count < 15:
        flags.append({
            "StudyInstanceUID": row["StudyInstanceUID"], "reason": "Very short translated report",
            "detail": text[:80], "report_snippet": text[:120],
        })

    was_missing_map = {
        target: pd.isna(raw_labels.loc[raw_labels["StudyInstanceUID"] == row["StudyInstanceUID"], target].values[0])
        for target in target_columns
    }
    if not any(was_missing_map.values()):
        continue

    has_global_normal = any(p.search(text) for p in GLOBAL_NORMAL_PATTERNS)

    doc = nlp.make_doc(text)
    spans = []
    for target, patterns in _COMPILED.items():
        if not was_missing_map[target]:
            continue
        for pattern in patterns:
            for match in pattern.finditer(text):
                span = doc.char_span(match.start(), match.end(), label=target, alignment_mode="expand")
                if span is not None:
                    spans.append(span)
    if not spans:
        continue
    doc.ents = filter_spans(spans)
    doc = nlp.get_pipe("sentencizer")(doc)
    doc = context(doc)
    doc = nlp.get_pipe("negex")(doc)

    for ent in doc.ents:
        target = ent.label_
        final_label = row[target]

        normal_phrase_match = any(p.search(text) for p in _NORMAL_COMPILED.get(target, []))
        if final_label == 0 and (normal_phrase_match or has_global_normal):
            continue

        medspacy_negated = ent._.is_negated or ent._.is_uncertain or ent._.is_hypothetical
        negspacy_negated = ent._.negex
        is_negated = medspacy_negated or negspacy_negated

        if final_label == 1 and is_negated:
            flags.append({
                "StudyInstanceUID": row["StudyInstanceUID"],
                "reason": f"Positive '{target}' but medspaCy/negspacy flagged negation",
                "detail": ent.sent.text[:160],
                "report_snippet": text[:120],
            })
        if final_label == 0 and not is_negated:
            flags.append({
                "StudyInstanceUID": row["StudyInstanceUID"],
                "reason": f"Negative '{target}' but medspaCy/negspacy found no negation",
                "detail": ent.sent.text[:160],
                "report_snippet": text[:120],
            })

flags_df = pd.DataFrame(flags).drop_duplicates(subset=["StudyInstanceUID", "reason"])
flags_df.to_csv("/kaggle/working/labels_to_review.csv", index=False)
print(f"Total flagged cases needing your review: {len(flags_df)}")
print(flags_df["reason"].value_counts())

In [ ]:
# ===== NOTEBOOK CELL 4c: INSPECT FLAGGED CASES =====
import pandas as pd

flags_df = pd.read_csv("/kaggle/working/labels_to_review.csv")
for reason, group in flags_df.groupby("reason"):
    print(f"\n===== {reason} ({len(group)} cases) =====")
    for _, row in group.head(3).iterrows():
        print(f"- {row['detail']}")

In [ ]:
# ===== NOTEBOOK CELL 3: LOSS FUNCTION =====
import torch.nn as nn
import torch.nn.functional as F

class FocalClassBalancedBCELoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, target_names=None):
        super().__init__()
        self.target_names = target_names
        self.gamma = gamma
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_term = (1 - p_t) ** self.gamma
        per_target_loss = (focal_term * bce).mean(dim=0)
        return {"loss": per_target_loss.mean(), "per_target_loss": per_target_loss.detach()}
print("Loss class defined.")

In [ ]:
# ===== NOTEBOOK CELL 5: EXPERT PREDICTION HEADS =====
import torch.nn as nn
class ExpertPredictionHeads(nn.Module):
    def __init__(self, in_dim, target_names, hidden_dim=64):
        super().__init__()
        self.target_names = target_names
        self.heads = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden_dim, 1))
            for name in target_names
        })
    def forward(self, features):
        return torch.cat([self.heads[name](features) for name in self.target_names], dim=1)
print("Expert heads class defined.")

In [ ]:
# ===== NOTEBOOK CELL 6: ATTENTION SLICE POOLING =====
class AttentionSlicePooling(nn.Module):
    def __init__(self, feature_dim, hidden_dim=64):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))
    def forward(self, slice_features, slice_mask):
        scores = self.attention(slice_features).squeeze(-1)
        scores = scores.masked_fill(slice_mask == 0, float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=1), nan=0.0)
        pooled = torch.sum(slice_features * weights.unsqueeze(-1), dim=1)
        return pooled, weights
print("Attention pooling class defined.")

In [ ]:
# ===== NOTEBOOK CELL 7: FULL MODEL ARCHITECTURE =====
import torchvision.models as tv_models
from sklearn.feature_extraction.text import TfidfVectorizer

class SliceBackbone(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        base = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        old_conv = base.conv1
        base.conv1 = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size, stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            base.conv1.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        base.fc = nn.Identity()
        self.encoder = base
        self.project = nn.Linear(512, out_dim)
    def forward(self, x):
        return self.project(self.encoder(x))

def fit_tfidf_vectorizer(report_texts, max_features=300):
    v = TfidfVectorizer(max_features=max_features)
    v.fit(report_texts)
    return v

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, out_dim=64):
        super().__init__()
        self.proj = nn.Linear(vocab_size, out_dim)
    def forward(self, tfidf):
        return self.proj(tfidf)

class MultimodalKneeModel(nn.Module):
    def __init__(self, target_names, vocab_size, img_feat_dim=128, text_feat_dim=64):
        super().__init__()
        self.target_names = target_names
        self.backbone = SliceBackbone(out_dim=img_feat_dim)
        self.slice_pooling = AttentionSlicePooling(feature_dim=img_feat_dim)
        self.text_encoder = TextEncoder(vocab_size, out_dim=text_feat_dim)
        self.text_feat_dim = text_feat_dim
        self.fusion = nn.Linear(img_feat_dim + text_feat_dim, img_feat_dim)
        self.heads = ExpertPredictionHeads(in_dim=img_feat_dim, target_names=target_names)
    def encode_image(self, slice_stack, slice_mask):
        b, s, c, h, w = slice_stack.shape
        feats = self.backbone(slice_stack.view(b*s, c, h, w)).view(b, s, -1)
        return self.slice_pooling(feats, slice_mask)
    def forward(self, slice_stack, slice_mask, text_tfidf=None, use_text=True):
        img_features, attn_weights = self.encode_image(slice_stack, slice_mask)
        b = img_features.shape[0]
        if use_text and text_tfidf is not None:
            text_features = self.text_encoder(text_tfidf)
        else:
            text_features = torch.zeros(b, self.text_feat_dim, dtype=img_features.dtype, device=img_features.device)
        fused = self.fusion(torch.cat([img_features, text_features], dim=1))
        return self.heads(fused), attn_weights

dummy_vectorizer = fit_tfidf_vectorizer(["placeholder report text for shape-testing only"])
print("Full model architecture defined.")

In [ ]:
# ===== NOTEBOOK CELL 8: FULL REAL TRAINING — SEQUENTIAL SLICE COVERAGE =====
import os, time, gc, math
import numpy as np, pandas as pd, torch, torch.nn as nn, cv2, pydicom
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

KAGGLE_DATA = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files and "competitions" in root and "rsna-knee-abnormality-detection" in root:
        KAGGLE_DATA = Path(root)
        break

labels_df = pd.read_csv("/kaggle/working/train_labels_filled.csv")
target_columns = [c for c in labels_df.columns if c != "StudyInstanceUID"]
labels_df = labels_df.merge(train_full[["StudyInstanceUID", "Report_English"]], on="StudyInstanceUID", how="left")
labels_df["Report_English"] = labels_df["Report_English"].fillna("")

train_series = pd.read_csv(KAGGLE_DATA / "train_series.csv")
series_root = KAGGLE_DATA / "train_series"

TARGET_SIZE = 224
CHUNK_SIZE = 64

def get_real_file_list(study_uid, series_root, train_series_df):
    study_series = train_series_df[train_series_df["StudyInstanceUID"] == study_uid]
    all_files = []
    for _, row in study_series.iterrows():
        series_dir = series_root / study_uid / row["SeriesInstanceUID"]
        for fname in os.listdir(series_dir):
            all_files.append(series_dir / fname)
    if not all_files:
        raise ValueError(f"No real files found for study {study_uid}")
    return sorted(all_files)

def load_study_chunk(study_uid, series_root, train_series_df, chunk_index, chunk_size=CHUNK_SIZE):
    all_files = get_real_file_list(study_uid, series_root, train_series_df)
    num_chunks = math.ceil(len(all_files) / chunk_size)
    real_chunk_index = chunk_index % num_chunks
    start = real_chunk_index * chunk_size
    chunk_files = all_files[start:start + chunk_size]
    slices = []
    for dcm_path in chunk_files:
        try:
            dcm = pydicom.dcmread(dcm_path)
            arr = dcm.pixel_array.astype(np.float32)
            arr = cv2.resize(arr, (TARGET_SIZE, TARGET_SIZE))
            lo, hi = arr.min(), arr.max()
            if hi > lo:
                arr = (arr - lo) / (hi - lo)
            slices.append(arr)
        except Exception as e:
            print(f"Skipped unreadable file {dcm_path}: {e}")
    if not slices:
        raise ValueError(f"No real slices loaded for study {study_uid}, chunk {real_chunk_index}/{num_chunks}")
    return torch.from_numpy(np.stack(slices, axis=0)).unsqueeze(1).float()

def freeze_batchnorm(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False

n = len(labels_df)
pos_weight_values = [(n - int(labels_df[c].sum())) / int(labels_df[c].sum()) if labels_df[c].sum() > 0 else 1.0 for c in target_columns]
pos_weight_tensor = torch.tensor(pos_weight_values, dtype=torch.float32).to(device)

train_studies, val_studies = train_test_split(labels_df, test_size=0.10, random_state=0)

vectorizer = fit_tfidf_vectorizer(train_studies["Report_English"].tolist(), max_features=300)
vocab_size = len(vectorizer.vocabulary_)

model = MultimodalKneeModel(target_names=target_columns, vocab_size=vocab_size).to(device)
loss_fn = FocalClassBalancedBCELoss(pos_weight=pos_weight_tensor, target_names=target_columns).to(device)

MAX_EPOCHS = 10
PATIENCE = 3
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

best_avg_auc = -1.0
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    freeze_batchnorm(model.backbone.encoder)
    losses = []
    t0 = time.time()
    for i, (_, row) in enumerate(train_studies.iterrows()):
        try:
            ss = load_study_chunk(row["StudyInstanceUID"], series_root, train_series, chunk_index=epoch).to(device).unsqueeze(0)
        except Exception:
            continue
        mask = torch.ones(1, ss.shape[1]).to(device)
        labels = torch.tensor([row[c] for c in target_columns], dtype=torch.float32).unsqueeze(0).to(device)
        text_vec = vectorizer.transform([row["Report_English"]]).toarray()
        text_tfidf = torch.tensor(text_vec, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        logits, _ = model(ss, mask, text_tfidf=text_tfidf, use_text=True)
        loss = loss_fn(logits, labels)["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for _, row in val_studies.iterrows():
            try:
                ss = load_study_chunk(row["StudyInstanceUID"], series_root, train_series, chunk_index=0).to(device).unsqueeze(0)
            except Exception:
                continue
            mask = torch.ones(1, ss.shape[1]).to(device)
            logits, _ = model(ss, mask, use_text=False)
            all_probs.append(torch.sigmoid(logits).cpu().numpy()[0])
            all_labels.append(row[target_columns].values.astype(float))
    all_probs, all_labels = np.array(all_probs), np.array(all_labels)

    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS} — train loss: {np.mean(losses):.4f} — {round(time.time()-t0)}s")
    aucs = []
    for idx, target in enumerate(target_columns):
        try:
            auc = roc_auc_score(all_labels[:, idx], all_probs[:, idx])
        except ValueError:
            auc = float("nan")
        aucs.append(auc)
        print(f"  {target}: AUC = {auc:.4f}")
    avg_auc = np.nanmean(aucs)
    print(f"  Average AUC: {avg_auc:.4f}")

    torch.save(model.state_dict(), f"/kaggle/working/model_epoch{epoch+1}.pt")
    if avg_auc > best_avg_auc:
        best_avg_auc = avg_auc
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "/kaggle/working/model_best.pt")
        print(f"  New best model saved (epoch {epoch+1}, AUC {avg_auc:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping — no improvement for {PATIENCE} epochs. Best epoch: {best_epoch} (AUC {best_avg_auc:.4f})")
            break

print(f"\nTraining complete. Best model: epoch {best_epoch}, AUC {best_avg_auc:.4f}")